In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from source_database.db_handler import DBConnection
import plotly.express as px

In [ ]:
db_handler = DBConnection()

# Query data;
query = {
    'gravatai_1':   'SELECT * FROM station_gravatai_1',
    'guaiba_1':     'SELECT * FROM station_guaiba_1', 
    'guaiba_2':     'SELECT * FROM station_guaiba_2',
    'sinos_1':      'SELECT * FROM station_sinos_1', 
    'sinos_2':      'SELECT * FROM station_sinos_2',
    'sinos_3':      'SELECT * FROM station_sinos_3',
    'taquari_1':    'SELECT * FROM station_taquari_1',
    }
dataframe = db_handler.run(query=query)

# Open query into single dfs;
gravatai_1  =   dataframe.get('gravatai_1')
guaiba_1    =   dataframe.get('guaiba_1')
guaiba_2    =   dataframe.get('guaiba_2')
sinos_1     =   dataframe.get('sinos_1')
sinos_2     =   dataframe.get('sinos_2')
sinos_3     =   dataframe.get('sinos_3')
taquari_1   =   dataframe.get('taquari_1')

In [ ]:
def outlier_visualization(df, df_features_clean, original_indices, feature_cols,
                          ecod_predictions, ecod_scores, ecod_threshold,
                          pca_predictions, pca_scores, pca_threshold,
                          combined_predictions):
    """
    Create visualizations for outlier detection results;

    Parameters:
        df (pd.DataFrame): Original dataframe;
        df_features_clean (pd.DataFrame): Cleaned features used for detection;
        original_indices (pd.Index): Indices of cleaned data in original df;
        feature_cols (list): List of feature column names;
        ecod_predictions (np.array): ECOD outlier predictions (0=normal, 1=outlier);
        ecod_scores (np.array): ECOD outlier scores;
        ecod_threshold (float): ECOD decision threshold;
        pca_predictions (np.array): PCA outlier predictions;
        pca_scores (np.array): PCA outlier scores;
        pca_threshold (float): PCA decision threshold;
        combined_predictions (np.array): Combined outlier predictions;
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.decomposition import PCA as sklearn_PCA
    
    print("\n" + "-"*80)
    print("CREATING VISUALIZATIONS")
    print("-"*80)
    
    sns.set_style("whitegrid")
    plt.rcParams['figure.facecolor'] = 'white'
    
    # 1. Outlier Scores Distribution;
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    axes[0].hist(ecod_scores[ecod_predictions == 0], bins=50, alpha=0.7, label='Normal', color='blue')
    axes[0].hist(ecod_scores[ecod_predictions == 1], bins=50, alpha=0.7, label='Outlier', color='red')
    axes[0].axvline(ecod_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {ecod_threshold:.3f}')
    axes[0].set_xlabel('Outlier Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('ECOD Outlier Scores Distribution', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(pca_scores[pca_predictions == 0], bins=50, alpha=0.7, label='Normal', color='blue')
    axes[1].hist(pca_scores[pca_predictions == 1], bins=50, alpha=0.7, label='Outlier', color='red')
    axes[1].axvline(pca_threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {pca_threshold:.3f}')
    axes[1].set_xlabel('Outlier Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('PCA Outlier Scores Distribution', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_scores_distribution.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_scores_distribution.png")
    plt.close()
    
    # 2. PCA 2D Projection;
    pca_2d = sklearn_PCA(n_components=2)
    features_2d = pca_2d.fit_transform(df_features_clean)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for idx, (predictions, title) in enumerate([(ecod_predictions, 'ECOD'),
                                                  (pca_predictions, 'PCA'),
                                                  (combined_predictions, 'Combined')]):
        axes[idx].scatter(features_2d[predictions == 0, 0], features_2d[predictions == 0, 1],
                         c='blue', alpha=0.5, s=10, label='Normal')
        axes[idx].scatter(features_2d[predictions == 1, 0], features_2d[predictions == 1, 1],
                         c='red', alpha=0.8, s=30, label='Outlier', edgecolors='black', linewidths=0.5)
        axes[idx].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
        axes[idx].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
        axes[idx].set_title(f'{title} Outliers in PCA Space', fontsize=14, fontweight='bold')
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_pca_projection.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_pca_projection.png")
    plt.close()
    
    # 3. Feature-wise boxplots;
    n_features = len(feature_cols)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
    axes = axes.flatten() if n_features > 1 else [axes]
    
    for idx, feature in enumerate(feature_cols):
        normal_data = df_features_clean.loc[combined_predictions == 0, feature]
        outlier_data = df_features_clean.loc[combined_predictions == 1, feature]
        bp = axes[idx].boxplot([normal_data, outlier_data], labels=['Normal', 'Outlier'],
                               patch_artist=True, widths=0.6)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightcoral')
        axes[idx].set_ylabel('Value', fontsize=10)
        axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    for idx in range(n_features, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Feature Distribution: Normal vs Outliers', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_feature_boxplots.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_feature_boxplots.png")
    plt.close()
    
    # 4. Time series with outliers;
    if 'date' in df.columns:
        df_viz = df.loc[original_indices].copy()
        df_viz['combined_outlier'] = combined_predictions
        
        fig, axes = plt.subplots(len(feature_cols), 1, figsize=(16, len(feature_cols) * 3))
        if len(feature_cols) == 1:
            axes = [axes]
        
        for idx, feature in enumerate(feature_cols):
            axes[idx].plot(df_viz['date'], df_viz[feature], color='gray', alpha=0.5, linewidth=0.5, label='Normal')
            outlier_mask = df_viz['combined_outlier'] == 1
            if outlier_mask.sum() > 0:
                axes[idx].scatter(df_viz.loc[outlier_mask, 'date'], df_viz.loc[outlier_mask, feature],
                                 color='red', s=20, alpha=0.8, label='Outlier', zorder=5)
            axes[idx].set_ylabel(feature, fontsize=10)
            axes[idx].set_title(f'{feature} - Time Series with Outliers', fontsize=11, fontweight='bold')
            axes[idx].grid(True, alpha=0.3)
            if idx == 0:
                axes[idx].legend(fontsize=9)
            if idx < len(feature_cols) - 1:
                axes[idx].set_xticklabels([])
        
        axes[-1].set_xlabel('Date', fontsize=12)
        plt.suptitle('Time Series Analysis - Outliers Highlighted', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_timeseries.png', dpi=300, bbox_inches='tight')
        print("Saved: outlier_timeseries.png")
        plt.close()
    
    # 5. Feature correlation with outliers;
    fig, ax = plt.subplots(figsize=(10, 8))
    df_corr = df_features_clean.copy()
    df_corr['is_outlier'] = combined_predictions
    correlation_with_outlier = df_corr.corr()['is_outlier'].drop('is_outlier').sort_values(ascending=False)
    
    colors = ['red' if x > 0 else 'blue' for x in correlation_with_outlier.values]
    ax.barh(range(len(correlation_with_outlier)), correlation_with_outlier.values, color=colors, alpha=0.7)
    ax.set_yticks(range(len(correlation_with_outlier)))
    ax.set_yticklabels(correlation_with_outlier.index, fontsize=10)
    ax.set_xlabel('Correlation with Outlier Status', fontsize=12)
    ax.set_title('Feature Correlation with Outlier Detection', fontsize=14, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('/home/juju/Documents/Nivel_TCC/data/outlier_feature_correlation.png', dpi=300, bbox_inches='tight')
    print("Saved: outlier_feature_correlation.png")
    plt.close()
    
    print("\n" + "="*80)
    print("OUTLIER DETECTION COMPLETE - All visualizations saved")
    print("="*80 + "\n")

In [ ]:
# Generate visualizations;
outlier_visualization(
    df=df,
    df_features_clean=df_features_clean,
    original_indices=original_indices,
    feature_cols=feature_cols,
    ecod_predictions=ecod_predictions,
    ecod_scores=ecod_scores,
    ecod_threshold=ecod_detector.threshold_,
    pca_predictions=pca_predictions,
    pca_scores=pca_scores,
    pca_threshold=pca_detector.threshold_,
    combined_predictions=combined_predictions
)